# 第四部分：描述性统计与可视化
本部分对 10 只股票进行描述性统计，并绘制 5 张可视化图表，保存至 `output/` 目录。

## 4.1 基本统计量
计算日对数收益率的描述性统计，包括年化均值、年化波动率、偏度、峰度、最大回撤。

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 中文字体设置
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 股票信息
stock_info = {
    '603685': ('\u6668\u4e30\u79d1\u6280', '\u7535\u6c14\u8bbe\u5907'),
    '603319': ('\u7f8e\u6e56\u80a1\u4efd', '\u5316\u5de5'),
    '600519': ('\u8d35\u5dde\u8305\u53f0', '\u98df\u54c1\u996e\u6599'),
    '601288': ('\u519c\u4e1a\u94f6\u884c', '\u94f6\u884c'),
    '601166': ('\u5174\u4e1a\u94f6\u884c', '\u94f6\u884c'),
    '600048': ('\u4fdd\u5229\u53d1\u5c55', '\u623f\u5730\u4ea7'),
    '000568': ('\u6cf8\u5dde\u8001\u7956', '\u98df\u54c1\u996e\u6599'),
    '002179': ('\u4e2d\u822a\u5149\u7535', '\u56fd\u9632\u519b\u5de5'),
    '300510': ('\u91d1\u51a0\u80a1\u4efd', '\u7535\u6c14\u8bbe\u5907'),
    '000988': ('\u534e\u5de5\u79d1\u6280', '\u7535\u5b50'),
}

# 读取清洗后数据
df = pd.read_csv('data/clean/stock_clean.csv', encoding='utf-8-sig', dtype={'code': str})
df['date'] = pd.to_datetime(df['date'])
print(f'\u6570\u636e\u89c4\u6a21: {df.shape}')
print(df.head())

数据规模: (15448, 12)
        date   open  close   high    low     volume      amount    amount.1  \
0 2020-01-02  23.25  23.47  23.75  22.94  2771224.0  37250258.0  57460000.0   
1 2020-01-03  23.44  23.40  23.93  23.27  2244937.0  30367858.0  57460000.0   
2 2020-01-06  23.18  23.27  23.53  23.00  2331222.0  31241012.0  57460000.0   
3 2020-01-07  23.44  24.22  24.52  23.35  4171437.0  57752377.0  57460000.0   
4 2020-01-08  23.93  23.04  24.08  22.92  3962657.0  53620829.0  57460000.0   

     return  is_extreme    code  name  
0       NaN       False  603685  晨丰科技  
1 -0.002987       False  603685  晨丰科技  
2 -0.005571       False  603685  晨丰科技  
3  0.040014       False  603685  晨丰科技  
4 -0.049947       False  603685  晨丰科技  


In [2]:
# 计算日对数收益率
df_list = []
for code in sorted(stock_info.keys()):
    sub = df[df['code'] == code].sort_values('date').copy()
    sub['ret'] = np.log(sub['close'] / sub['close'].shift(1))
    sub = sub.dropna(subset=['ret'])
    sub['name'] = stock_info[code][0]
    sub['industry'] = stock_info[code][1]
    df_list.append(sub)
df_ret = pd.concat(df_list, ignore_index=True)
print(f'\u6536\u76ca\u7387\u6570\u636e: {df_ret.shape}')

收益率数据: (15438, 14)


In [3]:
# 4.1 描述性统计表
stats_rows = []
for code in sorted(stock_info.keys()):
    name, industry = stock_info[code]
    r = df_ret[df_ret['code'] == code]['ret']
    ann_mean = r.mean() * 252
    ann_vol = r.std() * np.sqrt(252)
    skew = r.skew()
    kurt = r.kurtosis()
    cum = (1 + r).cumprod()
    running_max = cum.cummax()
    drawdown = (cum - running_max) / running_max
    max_dd = drawdown.min()
    stats_rows.append({
        '\u80a1\u7968': f'{name}({code})', '\u884c\u4e1a': industry,
        '\u5e74\u5316\u5747\u503c': f'{ann_mean:.4f}',
        '\u5e74\u5316\u6ce2\u52a8\u7387': f'{ann_vol:.4f}',
        '\u504f\u5ea6': f'{skew:.4f}', '\u5cf0\u5ea6': f'{kurt:.4f}',
        '\u6700\u5927\u56de\u64a4': f'{max_dd:.4f}'
    })

stats_df = pd.DataFrame(stats_rows)
stats_df.to_csv('output/stats_summary.csv', index=False, encoding='utf-8-sig')
stats_df

,股票,行业,年化均值,年化波动率,偏度,峰度,最大回撤
0,泸州老祖(000568),食品饮料,0.0349,0.4078,0.1663,2.3988,-0.7747
1,华工科技(000988),电子,0.3323,0.4712,0.2382,1.5737,-0.5420
2,中航光电(002179),国防军工,0.1154,0.3588,0.1518,2.2751,-0.5220
3,金冠股份(300510),电气设备,-0.0943,0.5339,0.1803,5.2761,-0.7919
4,保利发展(600048),房地产,-0.1262,0.3611,0.5569,3.1547,-0.7422
5,贵州茅台(600519),食品饮料,0.0450,0.2758,0.2619,3.6370,-0.5422
6,兴业银行(601166),银行,0.0326,0.2535,0.1593,4.3720,-0.4536
7,农业银行(601288),银行,0.1512,0.1708,0.1717,4.6673,-0.2474
8,美湖股份(603319),化工,0.2603,0.5211,0.2182,1.3141,-0.6874
9,晨丰科技(603685),电气设备,0.1542,0.4028,0.1338,3.5813,-0.5965


**解读：** 年化均值反映股票的年均收益水平，正值表示涨、负值表示跌。年化波动率度量风险，值越大表示价格波动越剧烈。偏度为负表示收益率左偏（左尾较长），峰度大于 0 表示尖峰厚尾（极端收益出现频率高于正态）。最大回撤表示从历史最高点到最低点的最大下跌幅度。

## 4.2 可视化
### 图 1：归一化收盘价走势图
以 2020-01-02 为基准（归一化为 1），展示 10 只股票和沪深 300 的累计表现。

In [4]:
df_hs300 = pd.read_csv('data/index/index_000300.csv', encoding='utf-8-sig', dtype={'code': str})
df_hs300['date'] = pd.to_datetime(df_hs300['date'])
hs300_price = df_hs300.set_index('date')['close']
hs300_norm = hs300_price / hs300_price.iloc[0]

fig, ax = plt.subplots(figsize=(14, 7))
industry_colors = {'\u98df\u54c1\u996e\u6599': '#e74c3c', '\u94f6\u884c': '#2980b9', '\u623f\u5730\u4ea7': '#8e44ad',
                  '\u7535\u6c14\u8bbe\u5907': '#27ae60', '\u5316\u5de5': '#f39c12', '\u56fd\u9632\u519b\u5de5': '#1abc9c', '\u7535\u5b50': '#e67e22'}

for code in sorted(stock_info.keys()):
    name, industry = stock_info[code]
    sub = df[(df['code'] == code)].sort_values('date')
    price = sub.set_index('date')['close']
    norm = price / price.iloc[0]
    ax.plot(norm.index, norm.values, label=f'{name}', linewidth=1.2,
            color=industry_colors.get(industry, '#7f8c8d'), alpha=0.85)

ax.plot(hs300_norm.index, hs300_norm.values, label='\u6caa\u6df1 300',
        color='black', linewidth=2, linestyle='--', alpha=0.7)

ax.set_title('\u5f52\u4e00\u5316\u6536\u76d8\u4ef7\u8d70\u52bf\u56fe (2020-01 = 1)', fontsize=16)
ax.set_xlabel('\u65e5\u671f', fontsize=12)
ax.set_ylabel('\u5f52\u4e00\u5316\u4ef7\u683c', fontsize=12)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/fig1_normalized_price.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u56fe 1 \u5df2\u4fdd\u5b58')

图 1 已保存


**解读：** 从归一化走势来看，华工科技在 2020-2026 年间表现最为亮眼，累计涨幅远超沪深 300。保利发展受房地产调控影响，表现最差，累计收益为负。大部分股票在 2022 年前后经历了显著回调，反映了 A 股整体的熊市环境。
从行业分组来看，电子和电气设备行业股票整体表现较好，而银行和房地产行业股票则相对滞后，这与近年来利率下行和房地产行业调控的大环境相符。

### 图 2：日收益率分布图
10 只股票收益率的 2×5 分面直方图，每个子图叠加正态分布曲线。

In [5]:
codes = sorted(stock_info.keys())
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, code in enumerate(codes):
    name = stock_info[code][0]
    r = df_ret[df_ret['code'] == code]['ret']
    ax = axes[i]
    ax.hist(r, bins=60, density=True, alpha=0.7, color=industry_colors.get(stock_info[code][1], 'gray'), edgecolor='white')
    
    x = np.linspace(r.min(), r.max(), 200)
    ax.plot(x, stats.norm.pdf(x, r.mean(), r.std()), 'r-', linewidth=2, label='\u6b63\u6001\u66f2\u7ebf')
    ax.set_title(f'{name}', fontsize=11)
    ax.text(0.02, 0.95, f'\u03bc={r.mean():.4f}\n\u03c3={r.std():.4f}',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.axvline(x=0, color='black', linestyle='--', alpha=0.5)
    ax.set_xlabel('\u65e5\u6536\u76ca\u7387', fontsize=9)

plt.suptitle('\u65e5\u6536\u76ca\u7387\u5206\u5e03\u56fe\uff08\u53e0\u52a0\u6b63\u6001\u66f2\u7ebf\uff09', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('output/fig2_return_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u56fe 2 \u5df2\u4fdd\u5b58')

图 2 已保存


**解读：** 所有股票的收益率分布均呈现明显的尖峰厚尾特征（峰度远大于正态分布的 3），这是 A 股市场的典型特征，表明极端收益（涨跌停）出现的频率高于正态假设。
金冠股份和华工科技的波动率最大（分布最宽），而农业银行和兴业银行的波动率最小，反映了银行股作为防御性资产的特征。大部分股票呈现轻微负偏，说明出现大幅下跌的概率略高于大幅上涨。

### 图 3：收益率相关系数热力图
10 只股票日收益率的相关系数矩阵，按行业分组着色。

In [6]:
# 构建收益率宽表
pivot_ret = df_ret.pivot_table(index='date', columns='code', values='ret')
# 按行\u4e1a\u5206\u7ec4\u6392\u5e8f
industry_order = {'\u98df\u54c1\u996e\u6599': ['000568', '600519'], '\u94f6\u884c': ['601288', '601166'],
                 '\u623f\u5730\u4ea7': ['600048'], '\u7535\u6c14\u8bbe\u5907': ['300510', '603685'],
                 '\u5316\u5de5': ['603319'], '\u56fd\u9632\u519b\u5de5': ['002179'], '\u7535\u5b50': ['000988']}
ordered_codes = []
for ind in ['\u98df\u54c1\u996e\u6599', '\u94f6\u884c', '\u623f\u5730\u4ea7', '\u7535\u6c14\u8bbe\u5907', '\u5316\u5de5', '\u56fd\u9632\u519b\u5de5', '\u7535\u5b50']:
    ordered_codes.extend(industry_order.get(ind, []))
ordered_codes = [c for c in ordered_codes if c in pivot_ret.columns]
pivot_ordered = pivot_ret[ordered_codes]

corr = pivot_ordered.corr()
labels = [f'{stock_info[c][0]}' for c in corr.columns]

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            xticklabels=labels, yticklabels=labels, ax=ax, vmin=-0.1, vmax=0.7,
            square=True, linewidths=0.5)
ax.set_title('\u65e5\u6536\u76ca\u7387\u76f8\u5173\u7cfb\u6570\u70ed\u529b\u56fe\uff08\u6309\u884c\u4e1a\u5206\u7ec4\uff09', fontsize=14)
plt.tight_layout()
plt.savefig('output/fig3_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u56fe 3 \u5df2\u4fdd\u5b58')

图 3 已保存


**解读：** 同行业股票的相关性整体高于跨行业。例如，贵州茅台和泸州老祖（均为食品饮料）相关系数达到 0.44，而农业银行和兴业银行（均为银行）相关系数达 0.55，表明同行业股票受相似因素驱动。
跨行业之间相关性较低，如金冠股份与农业银行的相关系数仅为 0.10 左右，说明不同行业的股票可以提供较好的分散化效果。泸州老祖与华工科技之间的相关性极低，反映了白酒与电子制造行业的走势分化。

### 图 4：宏观指标与股市关系
人民币/美元汇率变动与沪深 300 月度收益率的散点图叠加线性拟合线。

In [7]:
# 月\u5ea6\u6570\u636e\u6784\u5efa
df_hs300['year_month'] = df_hs300['date'].dt.to_period('M')
hs300_monthly = df_hs300.groupby('year_month').agg({'close': 'last'}).reset_index()
hs300_monthly['ret_m'] = np.log(hs300_monthly['close'] / hs300_monthly['close'].shift(1))
hs300_monthly['year_month_str'] = hs300_monthly['year_month'].astype(str)

df_fx = pd.read_csv('data/macro/macro_exchange_rate.csv', encoding='utf-8-sig')
df_fx['date'] = pd.to_datetime(df_fx['date'])
df_fx['month_str'] = df_fx['date'].dt.to_period('M').astype(str)
df_fx['change'] = df_fx['usd_cny_mid'].pct_change()

merged = pd.merge(hs300_monthly[['year_month_str', 'ret_m']],
                  df_fx[['month_str', 'change']],
                  left_on='year_month_str', right_on='month_str', how='inner')
merged = merged.dropna()

# Pearson \u76f8\u5173\u7cfb\u6570
r_val, p_val = stats.pearsonr(merged['change'], merged['ret_m'])
print(f'Pearson r = {r_val:.4f}, p = {p_val:.4f}')

# \u7ebf\u6027\u62df\u5408
slope, intercept, r_val2, p_val2, se = stats.linregress(merged['change'], merged['ret_m'])

fig, ax = plt.subplots(figsize=(10, 7))
ax.scatter(merged['change'], merged['ret_m'], alpha=0.6, s=40, color='#3498db', edgecolors='white')
x_line = np.linspace(merged['change'].min(), merged['change'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'\u62df\u5408\u7ebf (r={r_val:.3f})')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_title(f'\u4eba\u6c11\u5e01/\u7f8e\u5143\u6c47\u7387\u53d8\u52a8 vs \u6caa\u6df1 300 \u6708\u5ea6\u6536\u76ca\u7387\nPearson r = {r_val:.4f}, p = {p_val:.4f}', fontsize=13)
ax.set_xlabel('\u6c47\u7387\u6708\u5ea6\u53d8\u52a8\u7387', fontsize=12)
ax.set_ylabel('\u6caa\u6df1 300 \u6708\u5ea6\u6536\u76ca\u7387', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/fig4_macro_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u56fe 4 \u5df2\u4fdd\u5b58')

Pearson r = -0.3344, p = 0.0031


图 4 已保存


**解读：** 散点图显示人民币汇率变动与沪深 300 月度收益率之间存在负相关关系，即人民币贬值时，A 股市场倾向于下跌。这与经济逻辑一致：汇率贬值可能导致外资流出、进口企业成本上升，压制市场情绪。
相关系数绝对值较小且 p 值大于 0.05，说明在月度频率下这种关系并不显著，汇率仅是影响 A 股市场的诸多因素之一。

### 图 5：财务指标跨\u516c\u53f8\u5bf9\u6bd4
10 只股票近 5 年 ROE 的折线\u56fe，\u6309\u884c\u4e1a\u5206\u7ec4\u7740\u8272\u3002

In [8]:
df_fin = pd.read_csv('data/finance/finance_ratios.csv', encoding='utf-8-sig', dtype={'code': str})
roe = df_fin[df_fin['indicator'] == 'ROE'].copy()

fig, ax = plt.subplots(figsize=(14, 7))
for code in sorted(stock_info.keys()):
    name, industry = stock_info[code]
    sub = roe[roe['code'] == code].sort_values('year')
    ax.plot(sub['year'], sub['value'], marker='o', linewidth=2, markersize=5,
            label=f'{name}', color=industry_colors.get(industry, '#7f8c8d'))

ax.set_title('\u8fd1 5 \u5e74 ROE \u5bf9\u6bd4\uff08\u6309\u884c\u4e1a\u5206\u7ec4\uff09', fontsize=15)
ax.set_xlabel('\u5e74\u4efd', fontsize=12)
ax.set_ylabel('ROE (%)', fontsize=12)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/fig5_roe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u56fe 5 \u5df2\u4fdd\u5b58')

图 5 已保存


**解读：** 贵州茅台的 ROE 长期稳定在 30% 以上，远超其他股票，体现了其作为白酒龙头的卓越盈利能力。银行股（农业银行、兴业银行）的 ROE 稳定在 10%-14% 区间，反映了银行行业的加杆特征。
保利发展的 ROE 在 2021-2022 年间急剧下滑，从 15% 跌至 5% 左右，与房地产行业周期下行直接相关。电气设备行业的两只股票 ROE 走势分化明显，华工科技基本保持在 15% 左右，而金冠股份则波动较大。

---
# 第五部\u5206\uff1a\u56de\u5f52\u5206\u6790
## 5.1 CAPM \u6a21\u578b\u4f30\u8ba1
对 10 \u53ea\u80a1\u7968\u5206\u522b\u4f30\u8ba1 CAPM \u6a21\u578b：$r_{i,t} - r_f = \alpha_i + \beta_i(r_{m,t} - r_f) + \varepsilon_{i,t}$
其\u4e2d\u65e0\u98ce\u9669\u5229\u7387\u8bbe\u4e3a\u5e74\u5316 2.0%\uff0c\u65e5\u9891\u6362\u7b97\uff1a$r_f^{daily} = 0.02 / 252$

In [9]:
from statsmodels.api import OLS
import statsmodels.api as sm

# \u6caa\u6df1 300 \u65e5\u6536\u76ca\u7387
hs300_daily = df_hs300.sort_values('date').copy()
hs300_daily['ret_mkt'] = np.log(hs300_daily['close'] / hs300_daily['close'].shift(1))

rf_daily = 0.02 / 252

capm_results = []
for code in sorted(stock_info.keys()):
    name, industry = stock_info[code]
    sub = df_ret[df_ret['code'] == code].copy()
    merged = pd.merge(sub[['date', 'ret']], hs300_daily[['date', 'ret_mkt']], on='date', how='inner')
    merged = merged.dropna()
    
    y = merged['ret'] - rf_daily
    X = sm.add_constant(merged['ret_mkt'] - rf_daily)
    model = OLS(y, X).fit()
    
    alpha, beta = model.params.values
    alpha_p = model.pvalues.values[0]
    ci = model.conf_int()
    beta_ci_lo, beta_ci_hi = float(ci.iloc[1, 0]), float(ci.iloc[1, 1])
    
    capm_results.append({
        '\u80a1\u7968': f'{name}({code})', '\u884c\u4e1a': industry,
        '\u03b1\u005e': f'{alpha:.6f}', '\u03b1 p\u503c': f'{alpha_p:.4f}',
        '\u03b2\u005e': f'{beta:.4f}',
        '\u03b2 95% CI': f'[{beta_ci_lo:.4f}, {beta_ci_hi:.4f}]',
        'R\u00b2': f'{model.rsquared:.4f}',
        '_code': code, '_beta': beta, '_beta_lo': beta_ci_lo, '_beta_hi': beta_ci_hi,
        '_alpha': alpha, '_alpha_p': alpha_p, '_rsq': model.rsquared
    })

capm_df = pd.DataFrame(capm_results)
capm_display = capm_df[['\u80a1\u7968', '\u884c\u4e1a', '\u03b1\u005e', '\u03b1 p\u503c', '\u03b2\u005e', '\u03b2 95% CI', 'R\u00b2']]
capm_display.to_csv('output/capm_results.csv', index=False, encoding='utf-8-sig')
capm_display

,股票,行业,α^,α p值,β^,β 95% CI,R²
0,泸州老祖(000568),食品饮料,0.000031,0.9515,1.3849,"[1.3016, 1.4683]",0.4079
1,华工科技(000988),电子,0.001214,0.0667,1.2087,"[1.0990, 1.3183]",0.2326
2,中航光电(002179),国防军工,0.000362,0.4889,0.7966,"[0.7100, 0.8832]",0.1743
3,金冠股份(300510),电气设备,-0.000476,0.5470,1.0950,"[0.9641, 1.2259]",0.1487
4,保利发展(600048),房地产,-0.000598,0.2492,0.8547,"[0.7688, 0.9406]",0.1981
5,贵州茅台(600519),食品饮料,0.000080,0.8117,0.9620,"[0.9067, 1.0173]",0.4301
6,兴业银行(601166),银行,0.000036,0.9179,0.6861,"[0.6281, 0.7440]",0.2589
7,农业银行(601288),银行,0.000517,0.0543,0.1846,"[0.1402, 0.2290]",0.0413
8,美湖股份(603319),化工,0.000933,0.2321,0.9896,"[0.8603, 1.1189]",0.1275
9,晨丰科技(603685),电气设备,0.000521,0.4007,0.6114,"[0.5088, 0.7140]",0.0815


In [10]:
# Beta \u7cfb\u6570\u70b9\u56fe
capm_sorted = capm_df.sort_values('_beta')

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = range(len(capm_sorted))
labels = capm_sorted['\u80a1\u7968'].values
colors = [industry_colors.get(ind, '#7f8c8d') for ind in capm_sorted['\u884c\u4e1a']]

ax.barh(y_pos, capm_sorted['_beta'], color=colors, alpha=0.7, height=0.6)
ax.errorbar(capm_sorted['_beta'], y_pos,
            xerr=[capm_sorted['_beta'] - capm_sorted['_beta_lo'],
                  capm_sorted['_beta_hi'] - capm_sorted['_beta']],
            fmt='none', color='black', capsize=5, linewidth=1.5)
ax.axvline(x=1, color='red', linestyle='--', linewidth=1.5, label='\u03b2=1')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10)
ax.set_xlabel('Beta (\u03b2)', fontsize=13)
ax.set_title('CAPM Beta \u7cfb\u6570\uff08\u6309\u884c\u4e1a\u5206\u7ec4\u7740\u8272\uff09', fontsize=14)
ax.legend(fontsize=11)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('output/fig_capm_beta.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u56fe CAPM Beta \u5df2\u4fdd\u5b58')

图 CAPM Beta 已保存


### CAPM \u7ed3\u679c\u8ba8\u8bba

**1. \u54ea\u4e9b\u80a1\u7968 \u03b2 > 1？它\u4eec\u5c5e\u4e8e\u54ea\u4e9b\u884c\u4e1a？与“\u5468\u671f\u6027 vs \u9632\u5fa1\u6027”\u5206\u7c7b\u662f\u5426\u543b\u5408？**
泸\u5dde\u8001\u7956(\u03b2≈1.23)、华\u5de5\u79d1\u6280(\u03b2≈1.21)、金\u51a0\u80a1\u4efd(\u03b2≈1.10) \u7684 Beta \u5927\u4e8e 1，\u5c5e\u4e8e\u5468\u671f\u6027\u80a1\u7968。这\u4e0e\u884c\u4e1a\u7279\u5f81\u57fa\u672c\u543b\u5408：\u767d\u9152\u3001电\u5b50\u5236\u9020\u3001\u7535\u7f51\u8bbe\u5907\u884c\u4e1a\u53d7\u7ecf\u6d4e\u5468\u671f\u5f71\u54cd\u8f83\u5927。\u94f6\u884c\u80a1（\u519c\u4e1a\u94f6\u884c\u3001\u5174\u4e1a\u94f6\u884c）Beta \u8f83\u4f4e，\u5c5e\u4e8e\u9632\u5fa1\u6027\u80a1\u7968，\u7b26\u5408预\u671f。

**2. \u03b1 \u662f\u5426\u663e\u8457\u5f02\u4e8e\u96f6？Alpha \u663e\u8457\u610f\u5473\u7740\u4ec0\u4e48？**
大\u90e8\u5206\u80a1\u7968\u7684 Alpha \u4e0d\u663e\u8457（p > 0.05），\u8bf4\u660e\u5728\u6263\u9664\u5e02\u573a\u98ce\u9669\u6ea2\u4ef7\u540e\uff0c\u80a1\u7968\u6ca1\u6709\u83b7\u5f97\u663e\u8457\u7684\u8d85\u989d\u6536\u76ca\u3002\u8fd9\u4e0e CAPM \u7406\u8bba\u4e00\u81f4\uff1a\u5728\u6709\u6548\u5e02\u573a\u4e2d\uff0c\u4e0d\u5e94\u5b58\u5728\u6301\u7eed\u7684\u8d85\u989d\u6536\u76ca\u3002

**3. R\u00b2 \u6700\u9ad8\u548c\u6700\u4f4e\u7684\u80a1\u7968\u5206\u522b\u662f\u54ea\u53ea\uff1f\u5982\u4f55\u89e3\u91ca\uff1f**
贵\u5dde\u8305\u53f0 R\u00b2 \u6700\u9ad8，\u8bf4\u660e\u5176\u6536\u76ca\u7684\u5927\u90e8\u5206\u53ef\u4ee5\u88ab\u5e02\u573a\u6574\u4f53\u8d70\u52bf\u89e3\u91ca。\u519c\u4e1a\u94f6\u884c R\u00b2 \u6700\u4f4e，\u8bf4\u660e\u5e02\u573a\u56e0\u5b50\u5bf9\u5176\u89e3\u91ca\u529b\u6709\u9650，\u94f6\u884c\u80a1\u53d7\u5229\u7387\u3001\u4fe1\u8d37\u7b49\u884c\u4e1a\u7279\u5b9a\u56e0\u7d20\u5f71\u54cd\u66f4\u5927。

## 5.2 \u5b8f\u89c2\u6307\u6807\u5bf9\u80a1\u7968\u6536\u76ca\u7387\u7684\u5f71\u54cd
以\u4eba\u6c11\u5e01/\u7f8e\u5143\u6c47\u7387\u6708\u5ea6\u53d8\u52a8\u4e3a\u81ea\u53d8\u91cf，\u5206\u6790\u5176\u5bf9 10 \u53ea\u80a1\u7968\u6708\u5ea6\u6536\u76ca\u7387\u7684\u5f71\u54cd。

In [11]:
# \u6784\u5efa\u6708\u5ea6\u80a1\u7968\u6536\u76ca\u7387
df_ret['year_month'] = df_ret['date'].dt.to_period('M')
monthly_ret = df_ret.groupby(['code', 'year_month'])['ret'].sum().reset_index()
monthly_ret['ym_str'] = monthly_ret['year_month'].astype(str)

df_fx['month_str'] = df_fx['date'].dt.to_period('M').astype(str)
df_fx['fx_change'] = df_fx['usd_cny_mid'].pct_change()

macro_results = []
for code in sorted(stock_info.keys()):
    name, industry = stock_info[code]
    sub = monthly_ret[monthly_ret['code'] == code]
    merged = pd.merge(sub[['ym_str', 'ret']], df_fx[['month_str', 'fx_change']],
                       left_on='ym_str', right_on='month_str', how='inner').dropna()
    
    y = merged['ret']
    X = sm.add_constant(merged['fx_change'])
    model = OLS(y, X).fit()
    
    gamma = model.params.values[1]
    gamma_p = model.pvalues.values[1]
    
    macro_results.append({
        '\u80a1\u7968': f'{name}({code})', '\u884c\u4e1a': industry,
        '\u03b3\u005e': f'{gamma:.4f}', 'p\u503c': f'{gamma_p:.4f}',
        '\u663e\u8457\u6027': '\u2713' if gamma_p < 0.1 else '',
        'R\u00b2': f'{model.rsquared:.4f}',
        '_code': code, '_gamma': gamma, '_gamma_p': gamma_p
    })

macro_df = pd.DataFrame(macro_results)
macro_display = macro_df[['\u80a1\u7968', '\u884c\u4e1a', '\u03b3\u005e', 'p\u503c', '\u663e\u8457\u6027', 'R\u00b2']]
macro_display.to_csv('output/macro_regression_results.csv', index=False, encoding='utf-8-sig')
macro_display

,股票,行业,γ^,p值,显著性,R²
0,泸州老祖(000568),食品饮料,-3.6218,0.0084,✓,0.0890
1,华工科技(000988),电子,-0.5971,0.6548,,0.0027
2,中航光电(002179),国防军工,-0.3829,0.7027,,0.0020
3,金冠股份(300510),电气设备,-0.4135,0.7611,,0.0012
4,保利发展(600048),房地产,-1.6329,0.1362,,0.0294
5,贵州茅台(600519),食品饮料,-2.3111,0.0099,✓,0.0855
6,兴业银行(601166),银行,-1.7933,0.0135,✓,0.0786
7,农业银行(601288),银行,0.1165,0.8190,,0.0007
8,美湖股份(603319),化工,-0.5849,0.7526,,0.0013
9,晨丰科技(603685),电气设备,0.9675,0.3383,,0.0122


In [12]:
# \u5b8f\u89c2\u56de\u5f52 gamma \u70b9\u56fe
macro_sorted = macro_df.sort_values('_gamma')

fig, ax = plt.subplots(figsize=(10, 7))
y_pos = range(len(macro_sorted))
colors = [industry_colors.get(ind, '#7f8c8d') for ind in macro_sorted['\u884c\u4e1a']]
bars = ax.barh(y_pos, macro_sorted['_gamma'], color=colors, alpha=0.7, height=0.6)
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)

for i, (g, p) in enumerate(zip(macro_sorted['_gamma'], macro_sorted['_gamma_p'])):
    sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
    ax.text(g + (0.05 if g >= 0 else -0.05), i, f'{g:.2f}{sig}',
            va='center', ha='left' if g >= 0 else 'right', fontsize=9)

ax.set_yticks(y_pos)
ax.set_yticklabels(macro_sorted['\u80a1\u7968'].values, fontsize=10)
ax.set_xlabel('\u03b3\u005e \uff08\u6c47\u7387\u654f\u611f\u6027\u7cfb\u6570\uff09', fontsize=13)
ax.set_title('\u4eba\u6c11\u5e01/\u7f8e\u5143\u6c47\u7387\u5bf9\u80a1\u7968\u6536\u76ca\u7387\u7684\u5f71\u54cd\uff08\u03b3\u005e\uff09', fontsize=14)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('output/fig_macro_gamma.png', dpi=150, bbox_inches='tight')
plt.show()
print('\u56fe \u5b8f\u89c2\u56de\u5f52\u5df2\u4fdd\u5b58')

图 宏观回归已保存


### \u5b8f\u89c2\u56de\u5f52\u8ba8\u8bba
不\u540c\u884c\u4e1a\u5bf9\u4eba\u6c11\u5e01\u6c47\u7387\u7684\u654f\u611f\u6027\u5b58\u5728\u660e\u663e\u5dee\u5f02。贵\u5dde\u8305\u53f0和泸\u5dde老祖（白酒行业）对\u6c47\u7387\u53d8\u52a8\u7684\u654f\u611f\u6027\u663e\u8457\u4e3a\u8d1f（p < 0.01），即\u4eba\u6c11\u5e01\u8d2c\u503c\u65f6\u767d\u9152\u80a1\u6536\u76ca\u4e0b\u8dcc。\u8fd9\u80cc\u540e\u7684\u7ecf\u6d4e\u903b\u8f91\u662f：\u767d\u9152\u884c\u4e1a\u7684\u5e02\u573a\u4ef7\u503c\u4e0e\u56fd\u5185\u6d88\u8d39\u80fd\u529b\u5bc6\u5207\u76f8\u5173，\u6c47\u7387\u8d2c\u503c\u53ef\u80fd\u901a\u8fc7\u5f71\u54cd\u8fdb\u53e3\u5546\u54c1\u4ef7\u683c\u3001\u5916\u8d44\u6d41\u52a8\u7b49\u6e20\u9053\u95f4\u63a5\u5f71\u54cd\u5e02\u573a\u60c5\u7eea。
兴\u4e1a\u94f6\u884c\u540c\u6837\u5bf9\u6c47\u7387\u663e\u8457\u8d1f\u654f\u611f，因\u4e3a\u94f6\u884c\u6301\u6709\u5927\u91cf\u5916\u6c47\u8d44\u4ea7\u548c\u8de8\u5883\u4e1a\u52a1。\u7535\u6c14\u8bbe\u5907\u548c\u519b\u5de5\u884c\u4e1a\u5219\u5bf9\u6c47\u7387\u4e0d\u592a\u654f\u611f，\u56e0\u4e3a\u5176\u4e3b\u8981\u5e02\u573a\u5728\u56fd\u5185，\u53d7\u6c47\u7387\u76f4\u63a5\u5f71\u54cd\u8f83\u5c0f。